In [ ]:
import time
import torch
from torch.utils.data import DataLoader
from torchvision.transforms import v2
from pathlib import Path

from benchmarking.bm_dl_only import benchmark_dataloader_only
from benchmarking.bm_train import benchmark_train_loop
from benchmarking.sweep_dl_configs import sweep_loader_configs

import sys
sys.path.append("..")
import v1.CarImageClass as CarImageClass
from v1.SSD_trainer import ConditionalIoUCrop, collate_detection, build_optimizer_and_scheduler
from v2.model_files.SSD_from_scratch import mySSD



device = "cuda" if torch.cuda.is_available() else "cpu"

# desktop, laptop, ubuntu
machine = 'ubuntu'

# Setup path to data folder
if machine == 'laptop':
    folder_path = Path(r"C:\self-driving-car\data")
elif machine == 'desktop':
    folder_path = Path(r"C:\Udacity_car_data\data")
elif machine == 'ubuntu':
    folder_path = Path(r"/mnt/c/Udacity_car_data/data")

train_path = folder_path / "train"
test_path = folder_path / "test"

In [12]:
# transforms
train_tfms = v2.Compose([
    v2.ToImage(),
    v2.ToDtype(torch.float32, scale=True),
    # v2.RandomZoomOut(fill=0, p=0.5),       # Zoom out hurts model performance

    ConditionalIoUCrop(min_area_frac=0.02,   # threshold between "large" and "small"
                       small_min_scale=0.4,
                       large_min_scale=0.7,
                       max_scale=1.0,
                       min_aspect_ratio=0.75,
                       max_aspect_ratio=1.33,
                       small_sampler_options=(0.0, 0.05, 0.1, 2.0),
                       large_sampler_options=(0.05, 0.1, 0.3, 2.0),
                       trials=10),

    v2.SanitizeBoundingBoxes(min_size=1.0),
    v2.RandomHorizontalFlip(p=0.5),
    v2.RandomPhotometricDistort(p=0.5),
    v2.Resize((300, 300), antialias=True),
    v2.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

test_tfms = v2.Compose([
    v2.ToImage(),
    v2.ToDtype(torch.float32, scale=True),
    v2.Resize((300, 300), antialias=True),
    v2.Normalize(mean=[0.485, 0.456, 0.406],
                 std=[0.229, 0.224, 0.225]),
])

# 993 images
train_data = CarImageClass.ImageClass(targ_dir=train_path, transform=train_tfms, file_pct=0.05, rand_seed=724, include_area=False)

In [13]:
benchmark_dataloader_only(dataset=train_data,
                          collate_fn=collate_detection,
                          batch_size=16,
                          num_workers=8,
                          pin_memory=True,
                          persistent_workers=True,
                          prefetch_factor=2,
                          multiprocessing_context=None,
                          shuffle=False,
                          warmup_batches=20,
                          measure_batches=100)

{'avg_batch_time_s': 0.11976780112000597,
 'batches_per_sec': 8.349489517620945,
 'samples_per_sec': 132.33940885429197,
 'median_batch_time_s': 0.0006837500259280205}

In [23]:
model = mySSD(class_to_idx_dict=train_data.class_to_idx, in_channels=3, variances=(0.1, 0.2)).to(device)

BATCH_SIZE = 16
NUM_WORKERS = 8
# Create DataLoader's
train_dataloader = DataLoader(train_data, 
                              batch_size=BATCH_SIZE, 
                              shuffle=True, 
                              num_workers=NUM_WORKERS,
                              persistent_workers=True,
                              prefetch_factor=2,
                              pin_memory=True,
                              collate_fn=collate_detection,
                              multiprocessing_context="spawn",
                              )

optimizer, scheduler = build_optimizer_and_scheduler(model=model,
                                                     train_dataloader=train_dataloader,
                                                     max_epochs=150,
                                                     warmup_epochs=5,
                                                     base_lr=0.003,
                                                     min_lr=1e-6,
                                                     momentum=0.9,
                                                     weight_decay=0.005)

scaler = torch.amp.GradScaler("cuda", enabled=True)

In [26]:
benchmark_train_loop(model=model,
                     dataset=train_data,
                     collate_fn=collate_detection,
                     optimizer=optimizer,
                     scheduler=scheduler,
                     scaler=scaler,
                     device=device,
                     batch_size=16,
                     num_workers=0,
                     pin_memory=False,
                     persistent_workers=False,
                     prefetch_factor=2,
                     multiprocessing_context=None,
                     shuffle=True,
                     warmup_steps=20,
                     measure_steps=100)

{'fetch_time_s': {'mean': 0.3865403972699187,
  'median': 0.38453819500000463,
  'p95': 0.45301358200049435},
 'h2d_time_s': {'mean': 0.004837045679996663,
  'median': 0.004782770000019809,
  'p95': 0.005962915999589313},
 'compute_time_s': {'mean': 0.2353921929399894,
  'median': 0.23738042400009363,
  'p95': 0.24464786699991237},
 'step_time_s': {'mean': 0.6267724919099783,
  'median': 0.6254391319998831,
  'p95': 0.6957530629997564},
 'samples_per_sec': 25.288282757432334,
 'batches_per_sec': 1.5954752528348475}

In [27]:
benchmark_train_loop(model=model,
                     dataset=train_data,
                     collate_fn=collate_detection,
                     optimizer=optimizer,
                     scheduler=scheduler,
                     scaler=scaler,
                     device=device,
                     batch_size=16,
                     num_workers=2,
                     pin_memory=False,
                     persistent_workers=False,
                     prefetch_factor=2,
                     multiprocessing_context=None,
                     shuffle=True,
                     warmup_steps=20,
                     measure_steps=100)

{'fetch_time_s': {'mean': 0.04189669013002458,
  'median': 0.03361126499976308,
  'p95': 0.05354146900026535},
 'h2d_time_s': {'mean': 0.008378241300006266,
  'median': 0.00810876850027853,
  'p95': 0.010256737999952747},
 'compute_time_s': {'mean': 0.2458982425800059,
  'median': 0.2463457674998608,
  'p95': 0.26374490300077014},
 'step_time_s': {'mean': 0.2961763584899927,
  'median': 0.2892399749998731,
  'p95': 0.32164415399984136},
 'samples_per_sec': 53.51541250898168,
 'batches_per_sec': 3.37636671980957}

In [28]:
benchmark_train_loop(model=model,
                     dataset=train_data,
                     collate_fn=collate_detection,
                     optimizer=optimizer,
                     scheduler=scheduler,
                     scaler=scaler,
                     device=device,
                     batch_size=16,
                     num_workers=4,
                     pin_memory=False,
                     persistent_workers=False,
                     prefetch_factor=2,
                     multiprocessing_context=None,
                     shuffle=True,
                     warmup_steps=20,
                     measure_steps=100)

{'fetch_time_s': {'mean': 0.03987878512002681,
  'median': 0.031653266999910556,
  'p95': 0.03951746800066758},
 'h2d_time_s': {'mean': 0.009109891430016432,
  'median': 0.008147533000283147,
  'p95': 0.013076479000119434},
 'compute_time_s': {'mean': 0.2577176630600934,
  'median': 0.24474245800047356,
  'p95': 0.3317498180003895},
 'step_time_s': {'mean': 0.3067094194699166,
  'median': 0.2857589624995853,
  'p95': 0.37703884199981985},
 'samples_per_sec': 51.677578169569834,
 'batches_per_sec': 3.260415026471283}

In [22]:
model = mySSD(class_to_idx_dict=train_data.class_to_idx, in_channels=3, variances=(0.1, 0.2))

steps_per_epoch = len(DataLoader(
    train_data,
    batch_size=16,
    shuffle=True,
    collate_fn=collate_detection,
))

def scheduler_ctor(optimizer):
    return torch.optim.lr_scheduler.OneCycleLR(
        optimizer,
        max_lr=1e-3,
        epochs=150,
        steps_per_epoch=steps_per_epoch,
    )

def optimizer_ctor(params):
    return torch.optim.SGD(params, lr=1e-3, momentum=0.9, weight_decay=0.005)

def scaler_ctor():
    return torch.amp.GradScaler("cuda", enabled=True)

sweep_loader_configs(model=model,
                     dataset=train_data,
                     collate_fn=collate_detection,
                     optimizer_ctor=optimizer_ctor,
                     scheduler_ctor=scheduler_ctor,
                     scaler_ctor=scaler_ctor,
                     batch_size=16,
                     device=device)

terminate called without an active exception
terminate called without an active exception
terminate called without an active exception
Exception in thread Thread-59 (_pin_memory_loop):
Traceback (most recent call last):
  File "/usr/lib/python3.12/threading.py", line 1073, in _bootstrap_inner
    self.run()
  File "/usr/lib/python3.12/threading.py", line 1010, in run
    self._target(*self._args, **self._kwargs)
  File "/home/eblackstone/repos/ssd-venv/lib/python3.12/site-packages/torch/utils/data/_utils/pin_memory.py", line 52, in _pin_memory_loop
    do_one_step()
  File "/home/eblackstone/repos/ssd-venv/lib/python3.12/site-packages/torch/utils/data/_utils/pin_memory.py", line 28, in do_one_step
    r = in_queue.get(timeout=MP_STATUS_CHECK_INTERVAL)
        ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/lib/python3.12/multiprocessing/queues.py", line 122, in get
    return _ForkingPickler.loads(res)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/eblackstone/repos/ss

KeyboardInterrupt: 